# Figure S1
Survival restarted at each period start. Variants:
- 90-day survivors from day 90
- 90-day survivors from onset (immortal time bias)
- all patients from onset
Each with full follow-up and cut at 1.5 and 1.75 years.

In [ ]:
import os
from enum import Enum
from itertools import combinations, product
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.transforms as transforms
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

In [ ]:
data_path = '../data/excel_4figures.xlsx'
output_dir = '../data/figures'
results_path = '../data/tables/figureS1_period_results.xlsx'

DAYS_PER_YEAR = 365.25
LANDMARK_DAYS = 90

# same-day deaths
MIN_DURATION_DAYS = 0.1

# (start, end, color); colors from original paper
PERIODS = {
    'Pre-pandemic (2018-2019)': (pd.Timestamp('2018-01-01'), pd.Timestamp('2020-01-01'), '#5a7a8c'),
    'Pandemic (2020-2021)': (pd.Timestamp('2020-01-01'), pd.Timestamp('2022-01-01'), '#b32f1e'),
    'Post-pandemic (2022-2024)': (pd.Timestamp('2022-01-01'), pd.Timestamp('2025-01-01'), '#95BF8F'),
}
PRE_PANDEMIC = 'Pre-pandemic (2018-2019)'

class FollowUp(Enum):
    FULL = 'full'
    HORIZON_1_5Y = 'horizon_1_5y'
    # 2 years - 90 days: end of pre-pandemic from day 90
    HORIZON_1_75Y = 'horizon_1_75y'


# None: until period end
HORIZON_YEARS = {FollowUp.FULL: None, FollowUp.HORIZON_1_5Y: 1.5, FollowUp.HORIZON_1_75Y: 1.75}

AT_RISK_YEARS = {
    FollowUp.FULL: [0, 1, 2, 3],
    FollowUp.HORIZON_1_5Y: [0, 0.5, 1, 1.5],
    FollowUp.HORIZON_1_75Y: [0, 0.5, 1, 1.5, 1.75],
}


class Variant(Enum):
    SURVIVORS_FROM_DAY_90 = 'survivors_from_day90'
    SURVIVORS_FROM_ONSET = 'survivors_from_onset'
    ALL_FROM_ONSET = 'all_from_onset'

In [ ]:
survival_df = pd.read_excel(data_path).dropna(how='all')

onset = pd.to_datetime(survival_df['Onset date'].astype(str), format='%Y%m%d')
death = pd.to_datetime(survival_df['5Y Death date'], format='%d/%m/%Y')

# from dates; Mortality_90d inconsistent
survived_90d = ~((death - onset).dt.days <= LANDMARK_DAYS)
patients_df = pd.DataFrame({'onset': onset, 'death': death, 'survived_90d': survived_90d})
patients_df['survived_90d'].value_counts()

In [ ]:
def period_survival(origin, death_date, end):
    # death on boundary date closes the earlier period
    death_in_period = death_date <= end
    stop = death_date.where(death_in_period, end)
    duration_years = (stop - origin).clip(lower=pd.Timedelta(days=MIN_DURATION_DAYS)) / pd.Timedelta(days=DAYS_PER_YEAR)
    return pd.DataFrame({'duration_years': duration_years, 'death': death_in_period})


def select_patients(variant):
    if variant == Variant.ALL_FROM_ONSET:
        return patients_df
    return patients_df[patients_df['survived_90d']]


def period_origin(patients, period, variant):
    """Time 0 and at-risk mask of each patient for a period."""
    start, end, _ = PERIODS[period]

    if period == PRE_PANDEMIC:
        origin = patients['onset']
        if variant == Variant.SURVIVORS_FROM_DAY_90:
            origin = origin + pd.Timedelta(days=LANDMARK_DAYS)

        # day 90 after period end: no time at risk
        return origin, origin < end

    # stroke before and alive at period start
    at_risk = (patients['onset'] < start) & (patients['death'].isna() | (patients['death'] > start))
    return pd.Series(start, index=patients.index), at_risk


def build_period_cohorts(variant, follow_up):
    patients = select_patients(variant)

    cohorts = {}
    for period, (_, end, _) in PERIODS.items():
        origin, at_risk = period_origin(patients, period, variant)
        origin = origin[at_risk]

        follow_up_end = pd.Series(end, index=origin.index)
        horizon_years = HORIZON_YEARS[follow_up]
        if horizon_years is not None:
            horizon_end = origin + pd.Timedelta(days=horizon_years * DAYS_PER_YEAR)
            follow_up_end = follow_up_end.where(follow_up_end < horizon_end, horizon_end)

        cohorts[period] = period_survival(origin, patients['death'][at_risk], follow_up_end)
    return cohorts


def pairwise_logrank(cohorts):
    rows = []
    for first, second in combinations(cohorts, 2):
        a, b = cohorts[first], cohorts[second]
        result = logrank_test(a['duration_years'], b['duration_years'],
                              event_observed_A=a['death'], event_observed_B=b['death'])
        rows.append({
            'period_a': first, 'patients_a': len(a), 'deaths_a': int(a['death'].sum()),
            'period_b': second, 'patients_b': len(b), 'deaths_b': int(b['death'].sum()),
            'p': round(result.p_value, 3),
        })
    return pd.DataFrame(rows)

def save_sheet(table, sheet_name):
    """Write one sheet, keeping the other notebook's sheets."""
    os.makedirs(os.path.dirname(results_path), exist_ok=True)
    if not os.path.exists(results_path):
        table.to_excel(results_path, sheet_name=sheet_name, index=False)
        return

    with pd.ExcelWriter(results_path, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        table.to_excel(writer, sheet_name=sheet_name, index=False)

In [ ]:
AT_RISK_HEADER_Y = -0.2
AT_RISK_ROW_SPACING = 0.055

# axes fraction
AT_RISK_LINE_X = (-0.12, -0.06)


def draw_at_risk_table(ax, fitters, at_risk_years):
    trans = transforms.blended_transform_factory(ax.transData, ax.transAxes)
    ax.text(AT_RISK_LINE_X[0], AT_RISK_HEADER_Y, 'Number at risk:', ha='left', transform=ax.transAxes)

    for row, (period, kmf) in enumerate(fitters.items(), start=1):
        y = AT_RISK_HEADER_Y - row * AT_RISK_ROW_SPACING
        _, _, color = PERIODS[period]
        ax.plot(AT_RISK_LINE_X, [y, y], color=color, transform=ax.transAxes, clip_on=False)

        for year in at_risk_years:
            n_at_risk = (kmf.durations >= year).sum()
            ax.text(year, y, n_at_risk, ha='center', va='center', transform=trans)


def plot_period_km(cohorts, follow_up):
    at_risk_years = AT_RISK_YEARS[follow_up]

    fig, ax = plt.subplots(1, 1, figsize=(7, 5.5))

    fitters = {}
    for period, cohort_df in cohorts.items():
        _, _, color = PERIODS[period]
        kmf = KaplanMeierFitter(label=f'{period} (n = {len(cohort_df)})')
        kmf.fit(cohort_df['duration_years'], event_observed=cohort_df['death'])
        kmf.plot(ax=ax, color=color, ci_show=False)
        fitters[period] = kmf

    # remove upper and right spine
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('black')
    ax.spines['left'].set_color('black')

    # Set axes
    ax.set_ylim(0, 1)
    ax.set_xlim(0, at_risk_years[-1])
    ax.set_xticks(at_risk_years)
    ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1])
    ax.set_yticklabels(['0', '20', '40', '60', '80', '100'])
    ax.yaxis.set_ticks_position('left')
    ax.set_ylabel('Survival rate (%)')
    ax.set_xlabel('Years from start of period')
    ax.grid(False)

    ax.legend(loc='lower right', frameon=False)
    draw_at_risk_table(ax, fitters, at_risk_years)

    # room for at-risk table
    plt.subplots_adjust(left=0.15, bottom=0.33)
    return fig

In [ ]:
os.makedirs(output_dir, exist_ok=True)

logrank_tables = []
for variant, follow_up in product(Variant, FollowUp):
    cohorts = build_period_cohorts(variant, follow_up)
    logrank_tables.append(pairwise_logrank(cohorts).assign(variant=variant.value, follow_up=follow_up.value))

    fig = plot_period_km(cohorts, follow_up)
    for extension in ['pdf', 'png']:
        fig.savefig(os.path.join(output_dir, f'figureS1_period_km_{variant.value}_{follow_up.value}.{extension}'),
                    dpi=600, bbox_inches='tight')

logrank_df = pd.concat(logrank_tables, ignore_index=True)
logrank_df = logrank_df[['variant', 'follow_up'] + [column for column in logrank_df if column not in ('variant', 'follow_up')]]
save_sheet(logrank_df, 'Log-rank (unadjusted)')
logrank_df